# שיעור 28 — פתרון מתועד — אימות ובחירת K
90 דקות מוצעות (שתי שעות לימוד). אחרי שיעורים 20–27; ידע קודם: KNN, הכנה מאימון ו־F1 לכל מחלקה.

**פתיחה:** דודו קיבל ציון גבוה אחרי שתרגל את אותן שאלות שוב ושוב. האם זה מעיד שיצליח בשאלות חדשות? גם מודל עלול להתאים היטב לדוגמאות האימון ולהתקשות בדוגמאות אחרות.
היום קובעים מראש מדד וערכי K, בוחרים ב־CV על אימון בלבד, ומתאימים מודל סופי לפני הערכת Test.

עוגן: [סיכום מפגש 8](../../09-ML-Fundamentals/Meeting-08-Didactic-Summary.md) ו[מחברת בחירת מודל](../../09-ML-Fundamentals/Meeting-08-Model-Selection-Guided.ipynb). זו התאמה דידקטית: בלי לולאת ציוני Test, עם השלמה בתוך Pipeline ועם macro-F1 קבוע מראש. חומרי המקור לא שונו.
**גבול ההדגמה:** Penguins כבר נבדק בכיתה/בחומרי ההוראה הקודמים. הקוד מדגים הפרדה נכונה בהרצה זו, אך הציון אינו מבחן עצמאי חדש של מאגר שלא הכרנו. בפרויקט חדש שומרים Test מתחילת העבודה ומדווחים אם השפיע על החלטות. אין טענה שהשיעורים הועברו בכיתה בלי דיווח המורה.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

## 1. חלוקה
Train פנימי לומד הכנה ומודל; Validation בוחר K; Test חיצוני מדווח ביצוע לאחר הבחירה. K הוא מספר שכנים, היפר־פרמטר. אין להשלים או לתקנן מראש את כל קבוצת האימון החיצונית, כי בכל סבב חלקה משמש לאימות.

In [ ]:
url = "https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df = pd.read_csv(url)
model_df = df.dropna(subset=["species"]).copy()
features = ["bill_length_mm","bill_depth_mm"]
X = model_df[features].copy()
y = model_df["species"].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
print("Outer Train:", len(X_train), "Outer Test:", len(X_test))
display(X_train.isna().sum())

## 2. תהליך CV
ה־Pipeline מתאים מחדש השלמה,תקנון ומודל באימון של כל קיפול. cross_validate מקבל X_train,y_train בלבד. השדה test_score שלו מתייחס לאימות הפנימי; הוא אינו מכיל את ציון Test החיצוני.

In [ ]:
# ההגדרות נקבעות לפני צפייה בהערכת Test
candidates = [1, 3, 5, 15, 51]
metric = "f1_macro"
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

# קוד תהליך נתון: המטרה להבין מה נכנס ל-CV ומה נשאר בחוץ
for k in candidates:
    pipe = make_pipeline(
        SimpleImputer(strategy="mean"),
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=k, weights="uniform", metric="euclidean")
    )
    fold_results = cross_validate(
        pipe, X_train, y_train,
        cv=cv, scoring=metric, return_train_score=True, n_jobs=1
    )
    # test_score כאן הוא Validation פנימי של הקיפול, לא ה-Test החיצוני
    results.append([
        k,
        fold_results["train_score"].mean(),
        fold_results["test_score"].mean(),
        fold_results["test_score"].std(ddof=0)
    ])

results_df = pd.DataFrame(results, columns=["K", "train_mean", "cv_mean", "cv_std"])
display(results_df)

## 3. בחירה
השורה עם cv_mean הגבוה ביותר נבחרת לפי הכלל שהוגדר מראש. פער בין אימון לאימות יכול להצביע על חשש להתאמת יתר, אבל אינו הוכחה יחידה; אי אפשר לדרוש שהציונים ישתנו באופן מונוטוני עם K.

In [ ]:
# כלל שנקבע מראש: ממוצע CV הגבוה ביותר; בשוויון מדויק K הקטן יותר
# המיון והגישה לשורה הם קטע עזר נתון
ranked = results_df.sort_values(["cv_mean", "K"], ascending=[False, True])
best_k = int(ranked.iloc[0]["K"])
print("Selected K by CV only:", best_k)

# מתאימים צינור חדש לכל קבוצת האימון החיצונית
final_model = make_pipeline(
    SimpleImputer(strategy="mean"),
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=best_k, weights="uniform", metric="euclidean")
)
final_model.fit(X_train, y_train)

## 4. הערכה
המודל הסופי הותאם מחדש לכל Train, ולא נבחר מבין חמשת מודלי הקיפולים. כעת מפיקים תחזיות פעם אחת, ומדווחים על אותה סדרת תחזיות בכמה מדדים.

In [ ]:
# רק לאחר בחירת K והתאמה מחדש: הערכה אחת על Test החיצוני
predictions = final_model.predict(X_test)
test_accuracy = accuracy_score(y_test, predictions)
labels_species = ["Adelie", "Chinstrap", "Gentoo"]
report = classification_report(
    y_test, predictions, labels=labels_species,
    output_dict=True, zero_division=0
)
report_table = pd.DataFrame(report).T
display(report_table.loc[labels_species, ["precision", "recall", "f1-score", "support"]])
test_macro_f1 = report["macro avg"]["f1-score"]
print("CV selection metric:", metric)
print("Selected model Test macro-F1:", test_macro_f1)
print("Selected model Test Accuracy:", test_accuracy)

## 5. כרטיס ניסוי ומגבלות
מועמדים [1,3,5,15,51],macro-F1, חלוקה 75%/25%,seed=17 ו־stratify, חמישה קיפולים עם seed=42. ההכרעה היא ממוצע CV גבוה ביותר ושוויון לפי K קטן. המדד ותהליך ההכרעה נבחרו לפני Test; טבלת התוצאות קובעת את K בפועל.
הכנה על כל X_train לפני CV תשתמש במידע מתוך רשומות האימות הפנימי לחישוב ממוצעים ותקנון. Pipeline בתוך CV מונע זאת. cv_std מתאר פיזור חמשת הציונים ואינו רווח סמך; הקיפולים גם אינם חמישה ניסויים עצמאיים לחלוטין.
Penguins כבר נראה בחומרים הקודמים, לכן זו הדגמת תהליך ולא אומדן עצמאי חדש של הכללה. שינוי seed אינו מחזיר מצב של נתונים שמעולם לא נראו.
כאשר יש כמה רשומות לאותו אדם, משאירים את כל הרשומות שלו באותו צד של החלוקה, וגם בקיפולים הפנימיים משתמשים בחלוקה לפי קבוצות. אחרת המודל עשוי לנצל זיהוי עקיף של האדם במקום להכליל לאחרים.
אין להשוות את ציון מחברת התרגול למודרכת כהוכחת עדיפות מאפיינים: גם החלוקה שונה. בפרויקט עתידי נועלים מראש את מתכון ההערכה ומתעדים חריגות.

### תוצאות ההרצה בעת הכנת הערכה
בתרגול נבחר K=15, עם ממוצע CV macro-F1 של כ־0.94070 וסטיית תקן תיאורית כ־0.03060. לאחר התאמה מחדש על 258 רשומות האימון, התקבלו ב־86 רשומות הבדיקה macro-F1 כ־0.95563 ו־Accuracy כ־0.96512. אין דרישה שציון Test יהיה זהה או נמוך מממוצע CV. אלה תוצאות הדגמה על Penguins המוכר, ולא הערכה בלתי תלויה חדשה. יש לקרוא מחדש את הפלט אם הנתונים או הספרייה השתנו; אין K מחייב מראש לתלמיד.